# Colab 2
## Frequent pattern mining

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf
from pyspark.ml.fpm import FPGrowth

In [3]:
conf = SparkConf().set("spark.ui.port", "4050").set("spark.executor.memory", "4g").set("spark.driver.memory", "4g")
sc = pyspark.SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/03 23:10:14 WARN Utils: Your hostname, panteliss-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.255.221.4 instead (on interface en0)
26/02/03 23:10:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/03 23:10:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/03 23:10:16 WARN Utils: Service 'SparkUI' could not bind on port 4050. Attempting port 4051.


In [4]:
products = spark.read.csv("../../assets/datasets/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("../../assets/datasets/order_products__train.csv", header=True, inferSchema=True)

### Join the items with orders in order to have list of item names (instead of just ids)

In [5]:
items_table = products.join(orders, on="product_id").groupby("order_id").agg(collect_list("product_id").alias("items"))

In [6]:
items_table.show()
fpGrowth = FPGrowth(itemsCol="items", minSupport = 0.01)
model = fpGrowth.fit(items_table)

+--------+--------------------+
|order_id|               items|
+--------+--------------------+
|     762|[21137, 41220, 15...|
|     844|[14992, 21405, 11...|
|     988|[45061, 28464, 12...|
|    1139|[24852, 21137, 34...|
|    1143|[42719, 42097, 36...|
|    1280|[48186, 49235, 23...|
|    1342|[13176, 30827, 14...|
|    1350|[12260, 25042, 38...|
|    1468|[24852, 46667, 30...|
|    1591|[17203, 44008, 48...|
|    1721|[38689, 27845, 22...|
|    1890|[432, 47626, 1317...|
|    1955|[10132, 13176, 16...|
|    2711|[9387, 17122, 273...|
|    2888|[30450, 41486, 46...|
|    3010|[2966, 47209, 131...|
|    3037|[19348, 19173, 6994]|
|    3179|[35221, 39322, 36...|
|    4036|[11758, 46979, 48...|
|    4092|[26209, 6615, 445...|
+--------+--------------------+
only showing top 20 rows


In [7]:
print(f'Frequent itemsets counted: {model.freqItemsets.count()}')
model.freqItemsets.show()

Frequent itemsets counted: 120
+--------------+-----+
|         items| freq|
+--------------+-----+
|       [24852]|18726|
|       [16797]| 6494|
|[16797, 24852]| 1948|
|       [22935]| 4290|
|       [42265]| 3597|
|        [5450]| 3103|
|       [37646]| 2809|
|       [21938]| 2521|
|       [39877]| 2337|
|       [19660]| 2225|
|        [8174]| 1980|
|       [22825]| 1893|
|       [16759]| 1742|
|       [46906]| 1527|
|       [40604]| 1449|
|       [20995]| 1361|
|       [13176]|15480|
|       [26209]| 6033|
|[26209, 24852]| 1331|
|[26209, 47626]| 1595|
+--------------+-----+
only showing top 20 rows


### Get the association rules by FPGrowth pyspark algorithm
Two cases with different minimum support for itemsets and confidence for the rules

In [8]:
print(f'Asscociation rules counted: {model.associationRules.count()}')
model.associationRules.show()

Asscociation rules counted: 0
+----------+----------+----------+----+-------+
|antecedent|consequent|confidence|lift|support|
+----------+----------+----------+----+-------+
+----------+----------+----------+----+-------+



In [9]:
fpGrowth2 = FPGrowth(itemsCol="items", minSupport = 0.001, minConfidence = 0.5)
model2 = fpGrowth2.fit(items_table)

In [10]:
print(f'Frequent itemsets counted: {model2.freqItemsets.count()}')
model2.freqItemsets.show()

Frequent itemsets counted: 4444


+--------------------+-----+
|               items| freq|
+--------------------+-----+
|             [24852]|18726|
|             [16797]| 6494|
|      [16797, 24852]| 1948|
|      [16797, 13176]|  777|
|      [16797, 21903]|  650|
|[16797, 21903, 24...|  217|
|      [16797, 47626]|  671|
|[16797, 47626, 24...|  281|
|      [16797, 47766]|  575|
|[16797, 47766, 24...|  267|
|      [16797, 47209]|  246|
|             [22935]| 4290|
|      [22935, 24852]|  679|
|      [22935, 13176]| 1000|
|      [22935, 21137]|  671|
|[22935, 21137, 13...|  258|
|      [22935, 21903]|  748|
|[22935, 21903, 24...|  142|
|[22935, 21903, 13...|  233|
|[22935, 21903, 21...|  163|
+--------------------+-----+
only showing top 20 rows


In [11]:
print(f'Asscociation rules counted: {model2.associationRules.count()}')
model2.associationRules.show()

Asscociation rules counted: 11
+--------------------+----------+------------------+------------------+--------------------+
|          antecedent|consequent|        confidence|              lift|             support|
+--------------------+----------+------------------+------------------+--------------------+
|      [22825, 47209]|   [13176]|0.5170454545454546|4.3824946411792345|0.001387099970276...|
|       [4605, 16797]|   [24852]|0.5357142857142857|3.7536332219526702|0.001143214261216...|
|[30391, 47209, 21...|   [13176]|          0.546875| 4.635330870478036|0.001066999977135...|
|      [27966, 47209]|   [13176]| 0.521099116781158| 4.416853618458589|0.004046978484707604|
|      [39928, 47209]|   [13176]|0.5459770114942529| 4.627719489738336|0.001448071397541327|
|      [35951, 47209]|   [13176]|0.5141065830721003| 4.357584667849303|0.001249914258930...|
|       [9839, 47209]|   [13176]|0.5048231511254019| 4.278897986822536|0.001196564260073623|
|       [8174, 47209]|   [13176]|0.5283